# BirdCLEF+ 2026 — Kaggle submission

5-fold MoE ensemble with Perch v2 embeddings and overlapping-window TTA.


In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    !pip install -q onnxruntime soundfile
    TEST_AUDIO_DIR = "/kaggle/input/competitions/birdclef-2026/test_soundscapes/"
    SAMPLE_SUB_PATH = "/kaggle/input/competitions/birdclef-2026/sample_submission.csv"
    MOE_DIR = "/kaggle/input/datasets/janpolobradorflaquer/birdclef-artifacts/bestmodel_e5_g1_wd1e-4_lr1e-3_ctx1_batchnorm/With Batch Norm"
    import glob
    PERCH_PATH = glob.glob("/kaggle/input/**/perch_v2_no_dft*.onnx", recursive=True)[0]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/BirdCLEF_Project")
    REPRO = DRIVE / "repro"
    TEST_AUDIO_DIR = str(DRIVE / "test_soundscapes")
    SAMPLE_SUB_PATH = str(REPRO / "data/metadata/sample_submission.csv")
    MOE_DIR = str(REPRO / "outputs/models/best_model")
    PERCH_PATH = str(DRIVE / "perch_v2_no_dft.onnx")

print(f"Runtime: {'Kaggle' if ON_KAGGLE else 'Colab/Drive'}")
print(f"MoE directory: {MOE_DIR}")
print(f"Perch model: {PERCH_PATH}")


In [ ]:
import concurrent.futures
import glob
import os

import numpy as np
import onnxruntime as ort
import pandas as pd
import soundfile as sf
from scipy.special import expit

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
target_columns = sample_sub.columns[1:].tolist()
NUM_CLASSES = len(target_columns)

so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

perch_session = ort.InferenceSession(PERCH_PATH, sess_options=so, providers=["CPUExecutionProvider"])
perch_in = perch_session.get_inputs()[0].name
perch_out = {o.name: i for i, o in enumerate(perch_session.get_outputs())}

moe_sessions = []
for moe_path in sorted(glob.glob(os.path.join(MOE_DIR, "*.onnx"))):
    moe_sessions.append(ort.InferenceSession(moe_path, sess_options=so, providers=["CPUExecutionProvider"]))
moe_in = moe_sessions[0].get_inputs()[0].name
print(f"Loaded {len(moe_sessions)} MoE models")


def read_and_pad_audio(path):
    y, _ = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim > 1:
        y = y.mean(axis=1)
    window_length = 5 * 32000
    if len(y) % window_length != 0 or len(y) == 0:
        y = np.pad(y, (0, window_length - (len(y) % window_length)))
    return y


test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.[oO][gG][gG]"))
predictions = []

if test_files:
    print(f"Processing {len(test_files)} files")
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        future_to_path = {executor.submit(read_and_pad_audio, p): p for p in test_files}
        for future in concurrent.futures.as_completed(future_to_path):
            path = future_to_path[future]
            filename = os.path.splitext(os.path.basename(path))[0]
            try:
                audio_1d = future.result()
                window_size = 160000
                hop_size = 80000
                chunks, start_times = [], []
                for start in range(0, len(audio_1d) - window_size + 1, hop_size):
                    chunks.append(audio_1d[start : start + window_size])
                    start_times.append(start / 32000)
                audio_batch = np.array(chunks, dtype=np.float32)
                embeddings = perch_session.run(None, {perch_in: audio_batch})[perch_out["embedding"]]
                context_batch = embeddings.reshape(embeddings.shape[0], 1536) if embeddings.ndim == 3 else embeddings
                fold_probs = []
                for sess in moe_sessions:
                    logits = sess.run(None, {moe_in: context_batch})[0]
                    fold_probs.append(expit(logits))
                avg_probs = np.mean(fold_probs, axis=0)
                window_preds = {}
                for i, start_time in enumerate(start_times):
                    end_time = start_time + 5.0
                    bins_to_update = {
                        int(np.ceil((start_time + 0.1) / 5.0) * 5),
                        int(np.ceil((end_time - 0.1) / 5.0) * 5),
                    }
                    for b in bins_to_update:
                        window_preds.setdefault(b, []).append(avg_probs[i])
                max_bin = len(audio_1d) // 32000
                for b in range(5, max_bin + 1, 5):
                    row_id = f"{filename}_{b}"
                    if b in window_preds:
                        final_prob = [float(x) for x in np.mean(window_preds[b], axis=0)]
                    else:
                        final_prob = [0.0] * NUM_CLASSES
                    predictions.append([row_id] + final_prob)
            except Exception as exc:
                print(f"Error on {filename}: {exc}")
                for b in range(5, 65, 5):
                    predictions.append([f"{filename}_{b}"] + [0.0] * NUM_CLASSES)

sub_df = pd.DataFrame(predictions, columns=["row_id"] + target_columns)
if not test_files:
    sub_df = sample_sub
sub_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")


In [ ]:
final_sub = pd.read_csv("submission.csv")
print(f"Shape: {final_sub.shape}")
print(f"NaN count: {final_sub.isnull().sum().sum()}")
print(f"Columns: {len(final_sub.columns)} (expected {NUM_CLASSES + 1})")
final_sub.head(3)
